# Climate Engine API to SPI

Pull a standardized index and the precipitation that produced it for three
points from the Climate Engine API, then recompute the index here from the same
precipitation.

The API key goes in the `CE_API_KEY` environment variable. It is sent in the
`Authorization` header and is never written into this notebook or its outputs.
Keys are issued by the Climate Engine team through their request form and expire
after 60 days; see `docs/earthengine-climate-engine.md`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from climate_indices import spi
from climate_indices.compute import Periodicity
from climate_indices.drought.indices import Distribution

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
from scripts.earthengine_climate_engine_pilot import (
    climate_engine_native,
    climate_engine_series,
    month_ends,
    parse_climate_engine_points,
    sample_at_month_ends,
)

key = os.environ.get("CE_API_KEY", "").strip()
assert key, "set CE_API_KEY to a Climate Engine API key"
print("key loaded, length", len(key))

## Ask the API for its own SPI-3

`distribution=nonparametric` is the rank-based option. `accumulation` is
expressed in the dataset's own unit, which is days for the daily gridMET
dataset, so a 3-month index is 90 days.

The endpoint returns a **daily** series: one value per day, each computed from
the trailing 90 days. That is a different cadence from this library's monthly
index, so the comparison below samples it at month ends.

In [ ]:
CALIBRATION = (1981, 2016)
WINDOW = ("1981-01-01", "2024-12-31")

ce_index = climate_engine_series(key, 3, "spi")
ce_points = parse_climate_engine_points(ce_index["response"], "spi")
print("request:", {k: v for k, v in ce_index["request"].items() if k != "coordinates"})
for point, values in ce_points.items():
    print(f"point {point}: {len(values)} daily values, {values.index[0].date()} to {values.index[-1].date()}")
ce_spi = ce_points[(-84.4, 33.0)]
ce_spi.head()

## Ask the API for the raw precipitation

The native endpoint returns the dataset's own values, in millimetres per day for
gridMET. Summing each calendar month gives the input the NumPy API expects.

In [ ]:
ce_native = climate_engine_native(key, "GRIDMET", "pr", *WINDOW)
daily_precipitation = parse_climate_engine_points(ce_native["response"], "pr")[(-84.4, 33.0)]
print("daily rows:", len(daily_precipitation))

# Label each calendar-month total by its last day, so the series lines up with
# the month ends at which the API index is sampled.
monthly_precipitation = daily_precipitation.resample("MS").sum()
monthly_precipitation.index = monthly_precipitation.index + pd.offsets.MonthEnd(0)
monthly_precipitation = monthly_precipitation[monthly_precipitation.index <= pd.Timestamp(WINDOW[1])]
print("monthly mean: %.1f mm" % monthly_precipitation.mean())
assert 1.0 < monthly_precipitation.mean() < 500.0, "monthly total is not in millimetres; check the API unit"
monthly_precipitation.tail()


## Compute SPI-3 here from the same precipitation

The API's SPI-3 is fitted by rank over its own sample. This call fits a gamma
distribution over 1981-2016. Both describe the same precipitation with different
methods, so a difference is expected; what this shows is the pipeline, with no
conversion step between the API response and a `climate_indices` input.

In [ ]:
ours = spi(
    monthly_precipitation.to_numpy(dtype="float64"),
    3,
    Distribution.gamma,
    int(monthly_precipitation.index[0].year),
    CALIBRATION[0],
    CALIBRATION[1],
    Periodicity.monthly,
)
ours_series = pd.Series(ours, index=monthly_precipitation.index, name="spi_gamma_here")

# Sample the API's daily series at the same month ends. Both sides then sit on an
# observation, and neither is interpolated.
window_months = month_ends(CALIBRATION[0], int(WINDOW[1][:4]))
api_month_ends = pd.Series(
    sample_at_month_ends(ce_spi, window_months),
    index=pd.DatetimeIndex([month.last_day for month in window_months]),
    name="spi_api_nonparametric",
)

paired = pd.concat([api_month_ends, ours_series], axis=1, join="inner").dropna()
correlation = paired.corr().iloc[0, 1]
difference = paired["spi_gamma_here"] - paired["spi_api_nonparametric"]
print(f"paired months: {len(paired)}   correlation: {correlation:.4f}")
print(f"mean difference (ours - api): {difference.mean():+.4f}   MAE: {difference.abs().mean():.4f}")
paired.tail()


In [ ]:
figure, axis = plt.subplots(figsize=(11, 4.5))
axis.plot(paired.index, paired["spi_api_nonparametric"], label="Climate Engine, nonparametric", linewidth=1.0)
axis.plot(paired.index, paired["spi_gamma_here"], label="climate_indices, gamma", linewidth=1.0, alpha=0.8)
axis.axhline(0.0, color="black", linewidth=0.8)
axis.set_ylabel("SPI-3")
axis.set_title("SPI-3 at one point, one precipitation series, two methods")
axis.legend(loc="lower left")
figure.tight_layout()

## What to watch for

- **The API returns a daily index, not a monthly one.** Comparing it to a
  monthly index requires choosing a sampling rule. This notebook takes the last
  observation on or before each month end, so the comparison lands on an
  observation rather than an interpolation.
- **Missing values arrive as `-9999`.** Left as a number, one sentinel would
  dominate a fit. The parser turns it into NaN.
- **The calibration window is a parameter of the request**, not a property of the
  data. Two callers who ask for different windows get different indices from the
  same precipitation.
- **The distribution matters more than the pipeline.** Rank-based and gamma fits
  do not have the same limits, so the two series differ most in the tails.